In [ ]:
# Configuração do ambiente

# Garante que o notebook está na raiz do projeto
%cd .. 

# Verifica o diretório atual (Linux/Mac)
# !pwd  

# Verifica o diretório atual (Windows)
!cd 

In [ ]:
# Importações necessárias 
from datetime import datetime

import pandas as pd

from src.config.auxiliares_ml import downcast_dataframe
from src.config.paths import DADOS_BRUTOS, DADOS_TRATADOS

In [ ]:
# Importando a base de dados

df = pd.read_csv(DADOS_BRUTOS)

In [ ]:
# Observando as primeiras linhas da base 

df.head(2)

In [ ]:
# Observando as informações da base 

df.info()

In [ ]:
# Observando os valores nulos da base

df.isnull().sum()

In [ ]:
# Filtrando os valores nulos de 'Postal Code'

df.loc[df['Postal Code'].isnull(), 'Postal Code'] = 5401

In [ ]:
# Criando uma função para corrigir os types das variáveis

def ajustar_tipos(df):
    df_corrigido = df.copy()
    
    for col in df_corrigido.columns:
        if col in ['Order Date', 'Ship Date']:
            # Manter essas colunas como tipo data
            df_corrigido[col] = pd.to_datetime(df_corrigido[col], errors='coerce')
        elif pd.api.types.is_numeric_dtype(df_corrigido[col]):
            # Se for numérico, verificar se pode ser convertido para inteiro
            if all(df_corrigido[col].dropna() % 1 == 0):
                df_corrigido[col] = df_corrigido[col].astype(int)
        elif pd.api.types.is_string_dtype(df_corrigido[col]):
            # Se for string e tiver muitas repetições, pode ser categorizado
            unique_ratio = df_corrigido[col].nunique() / len(df_corrigido[col])
            if unique_ratio < 0.5:  # Pode ajustar esse valor conforme necessário
                df_corrigido[col] = df_corrigido[col].astype('category')
        elif pd.api.types.is_object_dtype(df_corrigido[col]):
            # Verificar se é uma data
            try:
                df_corrigido[col] = df_corrigido[col].apply(lambda x: datetime.strptime(x, '%Y-%m-%d') if isinstance(x, str) else x)
            except (ValueError, TypeError):
                pass
    
    return df_corrigido

In [ ]:
# Utilizando a função 

df_corrigido = ajustar_tipos(df)
df_corrigido.dtypes

In [ ]:
# Fazendo o downcast da base 

df = downcast_dataframe(df)

In [ ]:
# Sanvando a base tratada na extensão .parquet

df.to_parquet(DADOS_TRATADOS)